# Lab 02 — Tool Calling

## Objective

Select and chain typed Python tools with validation and explicit errors.

## Introduction and concept

agent resolves operations through TOOLS and passes the validated requirements into the retrieval function.

## Real-world use case

A teacher needs to select questions from a small SQL question bank.

## Technology stack and dependencies

Python standard library. Use Python 3.11 or newer and a Jupyter kernel from the environment where you installed the [assignment requirements](../requirements.txt). Start the notebook from its lab directory.

## Architecture and workflow

Intent → Requirements tool → Question retrieval tool → Validation tool


## Step 1 — Implement the components

agent resolves operations through TOOLS and passes the validated requirements into the retrieval function.

In [1]:
BANK = {"SQL": ("Define a primary key", "Explain an inner join")}

def get_requirements(subject, count):
    if subject not in BANK:
        raise ValueError(f"Unknown subject: {subject}")
    if type(count) is not int or count < 1:
        raise ValueError("Count must be a positive integer")
    return {"subject": subject, "count": count}

def retrieve_questions(subject, count):
    if count > len(BANK[subject]):
        raise ValueError("Question bank has insufficient items")
    return BANK[subject][:count]

def validate_questions(questions):
    return bool(questions) and len(questions) == len(set(questions))

TOOLS = {
    "requirements": get_requirements,
    "retrieve": retrieve_questions,
    "validate": validate_questions,
}

def agent(intent, subject, count):
    trace = []
    try:
        requirements = TOOLS["requirements"](subject, count)
        trace.append("requirements")
        if intent == "inspect":
            return requirements, trace
        if intent != "build":
            raise ValueError("Unknown intent")
        trace.append("retrieve")
        questions = TOOLS["retrieve"](**requirements)
        trace.append("validate")
        if not TOOLS["validate"](questions):
            raise ValueError("Question validation failed")
        return questions, trace
    except ValueError as error:
        return {"error": str(error)}, trace

## Step 2 — Run sample inputs and inspect actual output

Use the SQL fixture with valid count, excessive count and an unsupported subject. Compare the returned values and tool traces.

In [2]:
def main():
    result, trace = agent("build", "SQL", 2)
    print(result, trace)
    assert len(result) == 2 and trace == ["requirements", "retrieve", "validate"]
    for args in [("inspect", "SQL", 1), ("build", "SQL", 3), ("build", "Math", 1)]:
        result, trace = agent(*args)
        print(args, result, trace)
    assert "error" in agent("build", "SQL", 0)[0]
    assert "error" in agent("build", "SQL", 3)[0]
    print("PASS: dispatch, chain and errors")

main()

('Define a primary key', 'Explain an inner join') ['requirements', 'retrieve', 'validate']
('inspect', 'SQL', 1) {'subject': 'SQL', 'count': 1} ['requirements']
('build', 'SQL', 3) {'error': 'Question bank has insufficient items'} ['requirements', 'retrieve']
('build', 'Math', 1) {'error': 'Unknown subject: Math'} []
PASS: dispatch, chain and errors


## Testing and validation

Use the SQL fixture with valid count, excessive count and an unsupported subject. Compare the returned values and tool traces. Assertions in the preceding cell check the demonstrated behavior and stop execution on a failed expectation.

## Limitations

The tool registry, intent rules and question bank are local fixtures.

## How this relates to ExamPilot

Implemented analogue. ExamPilot registers tools for question retrieval, validation, requirements and topics in backend/app/mcp/tools/exam_tools.py.

## Conclusion

Select and chain typed Python tools with validation and explicit errors. Use the trace and assertions to explain which decisions the workflow makes.